# 01 — Naive RAG (Baseline)

Este notebook construye el RAG más simple posible. Sin trucos, sin optimizaciones.
El objetivo es entender el flujo completo antes de añadir complejidad.

```
PDFs  →  Chunks  →  Embeddings  →  ChromaDB  →  Retrieve  →  Groq  →  Respuesta
```

**Antes de empezar:**
1. `cp .env.example .env` y añade tu `GROQ_API_KEY`
2. `python data/download_papers.py` para descargar los papers

In [1]:
# Setup: añade el root del proyecto al path para importar src/
import sys
from pathlib import Path

ROOT = Path().resolve().parent 
sys.path.insert(0, str(ROOT))

from dotenv import load_dotenv
load_dotenv(ROOT / '.env')

print('Root:', ROOT)
print('Setup OK')

Root: /Users/nicoglez/Desktop/Laboratory/rag-pipeline
Setup OK


## Paso 1 — Cargar y chunkear los PDFs

Cada PDF se divide en fragmentos de ~800 caracteres con 150 de solapamiento.
El solapamiento evita perder contexto cuando una idea queda partida entre dos chunks.

In [2]:
from src.ingestion.loader import load_pdfs, chunk_documents

# Carga todos los PDFs de data/raw/
docs = load_pdfs()
print(f'\nPáginas totales: {len(docs)}')

# Inspecciona una página
print('\n--- Ejemplo de página ---')
print(f'Paper: {docs[0].metadata["source"]}')
print(f'Página: {docs[0].metadata["page"]}')
print(f'Texto (primeros 300 chars):\n{docs[0].page_content[:300]}')

/Users/nicoglez/.pyenv/versions/3.12.13/lib/python3.12/site-packages/requests/__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (7.2.0)/charset_normalizer (3.4.6) doesn't match a supported version!
  warnings.warn(


  Cargando: attention_is_all_you_need.pdf
  Cargando: bert.pdf
  Cargando: chain_of_thought.pdf
  Cargando: gpt3.pdf
  Cargando: hyde.pdf
  Cargando: llama2.pdf
  Cargando: mixtral.pdf
  Cargando: rag_original.pdf
  Cargando: rag_survey.pdf
  Cargando: ragas.pdf

Total páginas cargadas: 298

Páginas totales: 298

--- Ejemplo de página ---
Paper: attention_is_all_you_need
Página: 0
Texto (primeros 300 chars):
Provided proper attribution is provided, Google hereby grants permission to
reproduce the tables and figures in this paper solely for use in journalistic or
scholarly works.
Attention Is All You Need
Ashish Vaswani∗
Google Brain
avaswani@google.com
Noam Shazeer∗
Google Brain
noam@google.com
Niki Par


In [3]:
# Divide en chunks
chunks = chunk_documents(docs)

# Inspecciona un chunk
print('\n--- Ejemplo de chunk ---')
sample = chunks[42]
print(f'Paper: {sample.metadata["source"]}')
print(f'Longitud: {len(sample.page_content)} chars')
print(f'Texto:\n{sample.page_content}')

Total chunks generados: 1665

--- Ejemplo de chunk ---
Paper: attention_is_all_you_need
Longitud: 764 chars
Texto:
In Table 3 rows (B), we observe that reducing the attention key size dk hurts model quality. This
suggests that determining compatibility is not easy and that a more sophisticated compatibility
function than dot product may be beneficial. We further observe in rows (C) and (D) that, as expected,
bigger models are better, and dropout is very helpful in avoiding over-fitting. In row (E) we replace our
sinusoidal positional encoding with learned positional embeddings [9], and observe nearly identical
results to the base model.
6.3 English Constituency Parsing
To evaluate if the Transformer can generalize to other tasks we performed experiments on English
constituency parsing. This task presents specific challenges: the output is subject to strong structural


## Paso 2 — Crear embeddings y guardar en ChromaDB

Convertimos cada chunk en un vector de ~1024 dimensiones (BGE-M3).
ChromaDB los guarda en disco en `chroma_db/` para no recalcular cada vez.

> **Primera vez**: tarda ~5-10 min (descarga modelo + genera embeddings para ~2000 chunks)

In [4]:
from src.ingestion.embedder import get_embeddings
from src.retrieval.vectorstore import build_vectorstore

embeddings = get_embeddings()  # carga/descarga BGE-M3
vectorstore = build_vectorstore(chunks, embeddings)

print(f'\nVector store listo con {vectorstore._collection.count()} vectores')

Cargando modelo de embeddings: BAAI/bge-m3
(Primera vez: descarga ~570 MB — después va al instante)


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/123 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/54.0 [00:00<?, ?B/s]

...

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/191 [00:00<?, ?B/s]

Construyendo vector store con 1665 chunks...
(Esto tarda un par de minutos la primera vez)
Vector store guardado en: /Users/nicoglez/Desktop/Laboratory/rag-pipeline/chroma_db

Vector store listo con 1665 vectores


## Paso 3 — Probar el retrieval

Antes de conectar el LLM, verificamos que el retrieval funciona solo.
Esto es importante para depurar: si el retrieval falla, la respuesta será mala
aunque el LLM sea perfecto.

In [5]:
query = 'How does the attention mechanism work in Transformers?'

retrieved = vectorstore.similarity_search(query, k=3)

print(f'Query: "{query}"')
print(f'\n{len(retrieved)} chunks recuperados:\n')
for i, doc in enumerate(retrieved):
    print(f'[{i+1}] Paper: {doc.metadata["source"]} | Página: {doc.metadata["page"]}')
    print(f'     {doc.page_content[:200]}...')
    print()

Query: "How does the attention mechanism work in Transformers?"

3 chunks recuperados:

[1] Paper: attention_is_all_you_need | Página: 4
     is similar to that of single-head attention with full dimensionality.
3.2.3 Applications of Attention in our Model
The Transformer uses multi-head attention in three different ways:
• In "encoder-deco...

[2] Paper: attention_is_all_you_need | Página: 1
     tion models in various tasks, allowing modeling of dependencies without regard to their distance in
the input or output sequences [2, 19]. In all but a few cases [27], however, such attention mechanis...

[3] Paper: attention_is_all_you_need | Página: 9
     7 Conclusion
In this work, we presented the Transformer, the first sequence transduction model based entirely on
attention, replacing the recurrent layers most commonly used in encoder-decoder archite...



## Paso 4 — Conectar el LLM y hacer preguntas

Ahora conectamos Groq (Llama 3.3 70B) para que genere la respuesta
usando los chunks como contexto.

In [1]:
from src.generation.chain import build_rag_chain, ask

chain = build_rag_chain(vectorstore, k=4)
print('Chain RAG creado')

ModuleNotFoundError: No module named 'src'

In [7]:
# Pregunta 1
question = 'What is the core innovation of the Transformer architecture?'
answer = ask(chain, question)
print(f'Q: {question}')
print(f'\nA: {answer}')

Q: What is the core innovation of the Transformer architecture?

A: La innovación central de la arquitectura Transformer es que reemplaza las capas recurrentes y convolucionales tradicionalmente utilizadas en modelos de secuencia a secuencia con un mecanismo de atención múltiple, lo que permite una mayor paralelización y reduce significativamente el tiempo de entrenamiento. En otras palabras, el Transformer se basa enteramente en un mecanismo de atención para calcular las representaciones de entrada y salida, sin utilizar redes neuronales recurrentes (RNN) o convolucionales.


In [8]:
# Pregunta 2 — sobre el paper original de RAG
question = 'What problem does RAG solve compared to standard language models?'
answer = ask(chain, question)
print(f'Q: {question}')
print(f'\nA: {answer}')

Q: What problem does RAG solve compared to standard language models?

A: RAG resuelve el problema de generar contenido incorrecto desde el punto de vista factual, ya que los modelos de lenguaje estándar pueden producir "alucinaciones" cuando se enfrentan a consultas que van más allá de sus datos de entrenamiento o requieren información actualizada. Al recuperar fragmentos de documentos relevantes de una base de conocimiento externa a través del cálculo de similitud semántica, RAG reduce efectivamente el problema de generar contenido incorrecto.


In [9]:
# Pregunta 3 — prueba libre
question = 'What are the main differences between BERT and GPT-3?'
answer = ask(chain, question)
print(f'Q: {question}')
print(f'\nA: {answer}')

Q: What are the main differences between BERT and GPT-3?

A: No encuentro esa información en los documentos disponibles. 

Sin embargo, puedo proporcionar algunas diferencias indirectas entre BERT y GPT-3 que se mencionan en el contexto:

* En SQuAD 2.0, GPT-3 mejora significativamente en un entorno de pocos disparos, mientras que BERT se menciona como un baseline.
* En RACE, GPT-3 se desempeña débilmente en comparación con el estado del arte, mientras que BERT se menciona como un modelo con el que se compara el desempeño de GPT-3.
* En algunos conjuntos de datos, como COPA y ReCoRD, GPT-3 logra un desempeño cercano al estado del arte en entornos de un solo disparo y pocos disparos, mientras que BERT se menciona como un modelo que se utiliza como referencia.

Es importante destacar que estas diferencias no son directas y se basan en comparaciones indirectas en diferentes conjuntos de datos y entornos de aprendizaje. Para obtener una respuesta más precisa, sería necesario tener acceso a

## Resumen

Hemos construido un RAG funcional en 4 pasos:

| Paso | Qué hace | Herramienta |
|------|----------|-------------|
| Cargar PDFs | Lee y parsea los archivos | PyPDFLoader |
| Chunkear | Divide en fragmentos de ~800 chars | RecursiveCharacterTextSplitter |
| Embeddings | Convierte texto en vectores | BGE-M3 (local) |
| Vector Store | Guarda y busca por similitud | ChromaDB |
| Generación | Responde usando el contexto | Llama 3.3 70B (Groq) |

**¿Qué limitaciones tiene este Naive RAG?**
- Vocabulary mismatch: si usas palabras distintas a las del paper, no encuentra bien
- Recupera solo los más similares en coseno, no los más útiles para responder
- Un solo ángulo de búsqueda por pregunta

→ Eso lo mejoramos en el **notebook 02**.